In [5]:
from sqlalchemy import URL, create_engine
import pandas as pd
import os
from dotenv import load_dotenv
load_dotenv()


server = "crm-data.database.windows.net"
database = os.getenv("DB_NAME")
username = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")

connection_url = URL.create(
    "mssql+pymssql",
    username=username,
    password=password,
    host=server,
    database=database
)

engine = create_engine(connection_url)

with engine.connect() as conn:
    print("Connection OK")

sch = pd.read_sql("""
SELECT *
FROM SCH
""", engine)

sch.head()

sci = pd.read_sql("""
SELECT *
FROM SCI
""", engine)

eq = pd.read_sql("""
SELECT *
FROM EQUIPMENT
""", engine)

prod = pd.read_sql("""
SELECT *
FROM PRODUCT
""", engine)

Connection OK


In [7]:
for name, df in {
    "SCH": sch,
    "SCI": sci,
    "EQ": eq,
    "PROD": prod
}.items():

    print(f"\n{name}")
    print(df.shape)
    print(df.head())


SCH
(303311, 15)
  ServiceConfirmationHeader EquipmentCode ServiceTechnicianCode MaterialCode  \
0                9000000033     E00705916              07004247      7410930   
1                9000000038     E01024195              07001740      7420830   
2                9000000042     E00702964              07001765     12015803   
3                9000000054     E01024807              07003558   Z3R3740685   
4                9000000103     E00639276              07001576     13002148   

                MaterialDescription HeaderStatusCode CreationTimestamp  \
0  ELCO LIGHTOIL BURNER EL 01A.8-1D            E0004        2016-01-07   
1                  EL 02A.16-1D (A)            E0004        2016-01-07   
2                        THISION 17            E0006        2016-01-07   
3                  BOILER 51-120 KW            E0004        2016-01-07   
4                  EK03.22 G-ZVT-T2            E0004        2016-01-07   

  EnergyTypeCode CategoryCode ProductTypeCode ProductTyp

In [8]:
sch.info()
sci.info()
eq.info()
prod.info()

<class 'pandas.DataFrame'>
RangeIndex: 303311 entries, 0 to 303310
Data columns (total 15 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   ServiceConfirmationHeader  303311 non-null  str           
 1   EquipmentCode              302934 non-null  str           
 2   ServiceTechnicianCode      303310 non-null  str           
 3   MaterialCode               302934 non-null  str           
 4   MaterialDescription        302934 non-null  str           
 5   HeaderStatusCode           303311 non-null  str           
 6   CreationTimestamp          303311 non-null  datetime64[us]
 7   EnergyTypeCode             302779 non-null  str           
 8   CategoryCode               302934 non-null  str           
 9   ProductTypeCode            302934 non-null  str           
 10  ProductType1               302779 non-null  str           
 11  ProductType2               302779 non-null  str           
 12 

In [9]:
sch.isnull().sum().sort_values(ascending=False)

SkillProfileCode             532
ProductType1                 532
EnergyTypeCode               532
ProductType2                 532
EquipmentCode                377
MaterialDescription          377
MaterialCode                 377
ProductTypeCode              377
CategoryCode                 377
ProductFamilyDescription     377
ServiceTechnicianCode          1
CreationTimestamp              0
HeaderStatusCode               0
ServiceConfirmationHeader      0
EquipmentAge                   0
dtype: int64

In [10]:
sci.isnull().sum().sort_values(ascending=False)

StatusCode                   10
ServiceConfirmationHeader     0
ServiceConfirmationItem       0
ItemTypeCode                  0
CreationTimestamp             0
StartWork                     0
EndWork                       0
ActualDuration                0
ProductCode                   0
dtype: int64

In [13]:
cluster_null = sch[
    sch['EnergyTypeCode'].isna() &
    sch['SkillProfileCode'].isna() &
    sch['ProductType1'].isna() &
    sch['ProductType2'].isna()
]
cluster_null.shape

cluster_prod = cluster_null.merge(
    prod,
    on='MaterialCode',
    how='left',
    suffixes=('_sch', '_prod')
)

cluster_prod[
    ['MaterialCode', 'MaterialDescription_prod']
].drop_duplicates()

cluster_prod.groupby(
    ['MaterialCode', 'MaterialDescription_prod']
).size().sort_values(
    ascending=False
).reset_index(
    name='NumCalls'
)

,MaterialCode,MaterialDescription_prod,NumCalls
0,12036131,EVACUATED TUBE EDF/AURON,38
1,3720473,REGLER LOGON B G2Z2 M.WANDGEHAEUSE,25
2,12056404,CONTROLLER LOGON SOL PLUS,23
3,11095593,STDRD THISION L 2-5-B-C,10
4,12087204,REGOLATORE RISC. LOGON B G2Z2 A PARETE,9
5,12056415,CONTROLLER LOGON SOL COMPACT,8
6,3722660,CONTROLLER LOGON B G2Z2 IN WALL HOUSING,6
7,12048319,SENSOR COLLECTOR 250°C RESISTANT,6
8,Z3R123999,INDUSTRIEBRENNER ALT,5
9,3024093,"CONNEC. SET 1FLAT COLL. (SOLATRON S,XP)",4


In [ ]:
# verify if the EnergyTypeCode_prod is not null for the MaterialCodes in cluster_prod
cluster_prod[
    [
        'MaterialCode',
        'MaterialDescription_prod',
        'EnergyTypeCode_prod'
    ]
].drop_duplicates()

,MaterialCode,MaterialDescription_prod,EnergyTypeCode_prod
0,3720473,REGLER LOGON B G2Z2 M.WANDGEHAEUSE,NaN
1,NaN,NaN,NaN
2,12087204,REGOLATORE RISC. LOGON B G2Z2 A PARETE,NaN
3,12013944,HEIZUNGSREGLER BG LOGON-M G2 AUSF.BJ,NaN
11,12001928,HEATER CONTROL LOGON-PLUS/ET-BG,NaN
21,12048319,SENSOR COLLECTOR 250°C RESISTANT,NaN
22,12036131,EVACUATED TUBE EDF/AURON,NaN
29,3024093,"CONNEC. SET 1FLAT COLL. (SOLATRON S,XP)",NaN
39,11095593,STDRD THISION L 2-5-B-C,NaN
44,11095243,STANDARD R30 2-5-A-C (K 01.1.0392),NaN


In [ ]:
cluster_377 = sch[
    sch['EquipmentCode'].isna()
]
cluster_377.shape
cluster_377.notna().sum().sort_values()

cluster_377['Year'] = cluster_377['CreationTimestamp'].dt.year

cluster_377['Year'].value_counts().sort_index()



Year
2016     27
2017     20
2018     26
2019     20
2020     18
2021     18
2022     21
2023     39
2024    103
2025     48
2026     37
Name: count, dtype: int64

In [21]:
cluster_377[['ServiceConfirmationHeader']]
cluster_377_sci = cluster_377.merge(
    sci,
    on='ServiceConfirmationHeader',
    how='left'
)
cluster_377_sci['ItemTypeCode'].value_counts()

ItemTypeCode
ZUNP    363
ZIND    363
ZSCR    186
ZRVM    158
ZTRH     98
ZSCS     83
ZTRZ     83
ZTRF     82
ZSWA     44
ZTHW     44
ZBRK     37
ZTFS     36
ZPWA     35
ZFSV     24
ZRVF     22
ZRSV     19
ZTRR     19
ZSCD     14
ZTRE     13
ZSCC      7
ZRVI      1
Name: count, dtype: int64

In [22]:
cluster_377[
    [
        'EquipmentCode',
        'MaterialCode'
    ]
].isna().all(axis=1).sum()

np.int64(377)

In [23]:
cluster_377[
    [
        'CreationTimestamp',
        'HeaderStatusCode',
        'ServiceTechnicianCode',
        'EnergyTypeCode',
        'SkillProfileCode'
    ]
].head(20)

,CreationTimestamp,HeaderStatusCode,ServiceTechnicianCode,EnergyTypeCode,SkillProfileCode
413,2016-01-18,E0004,07001765,NaN,NaN
1659,2016-01-28,E0004,07001765,NaN,NaN
6776,2016-03-30,E0004,07001583,NaN,NaN
7004,2016-04-04,E0004,07001583,NaN,NaN
9308,2016-05-02,E0004,07001765,NaN,NaN
9347,2016-05-02,E0004,07001765,NaN,NaN
9629,2016-05-04,E0004,07003558,NaN,NaN
11883,2016-05-31,E0004,07001765,NaN,NaN
12923,2016-06-13,E0004,07003558,NaN,NaN
16536,2016-08-01,E0006,07001765,NaN,NaN


In [24]:
eq.isnull().sum().sort_values(ascending=False)

InstallerCode       31108
StartUpDate         10915
WarrantyEndDate      9314
ManufacturerCode     1136
ConstructionYear        0
ProductCode             0
EquipmentCode           0
PowerCode               0
dtype: int64

In [ ]:
pd.crosstab(
    eq['ConstructionYear'],
    eq['InstallerCode'].isna(),
    normalize='index'
) * 100

In [26]:
(
    eq['StartUpDate'].isna()
    &
    eq['WarrantyEndDate'].isna()
).sum()

np.int64(9018)

In [ ]:
eq[
    eq['WarrantyEndDate']
    < eq['StartUpDate']
]

,EquipmentCode,ProductCode,ConstructionYear,InstallerCode,ManufacturerCode,PowerCode,WarrantyEndDate,StartUpDate
671,E00631515,1018582610,0,NaN,ELC,0.0,1986-06-27,2009-11-20
679,E00631578,7571434,2000,NaN,ELC,0.0,1991-07-10,2001-01-26
745,E00632190,11000543,2004,NaN,ELC,55.0,1988-09-26,2004-12-21
1128,E00634906,7571547,2005,NaN,ELC,0.0,1993-10-20,2005-02-20
1165,E00635193,3832630,2011,NaN,ELC,23.0,1993-12-15,2011-08-23
...,...,...,...,...,...,...,...,...
25323,E01013036,13010579,2002,NaN,ELC,0.0,2000-10-01,2002-10-14
25546,E01014476,7571553,0,NaN,ELC,0.0,2000-11-18,2002-09-02
25685,E01015210,13010595,2002,NaN,ELC,22.0,2000-10-08,2002-08-23
26419,E01019493,Z3R500000054,1998,NaN,ELC,23.0,1981-12-31,1999-09-01


In [37]:
invalid_eq = eq.loc[
    eq['WarrantyEndDate'] < eq['StartUpDate'],
    'EquipmentCode'
]
sch_invalid = sch[
    sch['EquipmentCode'].isin(invalid_eq)
]

len(sch_invalid)

print("Strange Equipment :", invalid_eq.nunique())
print("Related SCH Interventions:", len(sch_invalid))


Strange Equipment : 137
Related SCH Interventions: 969


In [47]:
invalid_prod = (
    invalid_eq_detail
    .merge(
        prod,
        left_on='ProductCode',
        right_on='MaterialCode',
        how='left'
    )
)
invalid_prod[
    ['ProductCode','MaterialDescription']
].drop_duplicates()

,ProductCode,MaterialDescription
0,1018582610,VECTRON 4V ZU UNON18
1,7571434,EK02.12 G-ZVU
2,11000543,EK02.6 L-T
3,7571547,EK01B.3 L-TH
4,3832630,CB-VE 1.34
...,...,...
109,13017834,VECTRON ECO L 01.34
128,4018657676,EL01A.28H-LNX/K27 E/A
129,4018657665,EL01A.28H-LNX/K-20 E/A
134,13010595,EK01B.28 L-NH OEKO-PLUS AUSF.BJ (SH113)


In [56]:
strange_eq = eq[
    eq["WarrantyEndDate"] < eq["StartUpDate"]
]
strange_sch = sch[
    sch["EquipmentCode"].isin(
        strange_eq["EquipmentCode"]
    )
]
strange_eq["ConstructionYear"].describe()

count     137.000000
mean     1885.014599
std       470.754073
min         0.000000
25%      1998.000000
50%      2003.000000
75%      2005.000000
max      2019.000000
Name: ConstructionYear, dtype: float64

In [57]:
prod.isnull().sum().sort_values(ascending=False)

WarengroupCode         644
ProductTypeCode         22
EnergyTypeCode          22
SkillProfileCode        22
MaterialCode             0
ProductFamilyCode        0
MaterialDescription      0
PowerMaxKW               0
PowerMinKW               0
PowerMaxLT               0
RefrigerantFluid         0
dtype: int64

In [58]:
cluster_22 = prod[
    prod['ProductTypeCode'].isna()
]

cluster_22.shape
(
    prod['ProductTypeCode'].isna()
    &
    prod['EnergyTypeCode'].isna()
    &
    prod['SkillProfileCode'].isna()
).sum()


np.int64(22)

In [59]:
cluster_22[
    [
        'MaterialCode',
        'MaterialDescription',
        'ProductFamilyCode',
        'WarengroupCode'
    ]
].sort_values('MaterialDescription')

,MaterialCode,MaterialDescription,ProductFamilyCode,WarengroupCode
816,3024093,"CONNEC. SET 1FLAT COLL. (SOLATRON S,XP)",10 99 132,2811.0
1197,3722660,CONTROLLER LOGON B G2Z2 IN WALL HOUSING,40 99 114,8117.0
524,12056415,CONTROLLER LOGON SOL COMPACT,10 99 132,8119.0
523,12056404,CONTROLLER LOGON SOL PLUS,10 99 132,8119.0
451,12036131,EVACUATED TUBE EDF/AURON,10 99 132,4130.0
338,12001928,HEATER CONTROL LOGON-PLUS/ET-BG,98 40 940,6116.0
2124,65001461,HEATER CONTROL MODULE LOGON B G2Z2,98 40 940,6116.0
370,12013944,HEIZUNGSREGLER BG LOGON-M G2 AUSF.BJ,98 40 940,6116.0
424,12029773,HEIZUNGSREGLER BG LOGON-SOL COMPACT,10 99 132,8124.0
513,12050255,HEIZUNGSREGLER RVA63.280/160,40 99 114,8117.0


found 2 clusters of rows in SCH with null values. 377 without equipment that must be discarded for the FTFR analysis. 532 with energy type, product type e skill profile empty that are all specific type of product, thermostat or similar things. Can be kept
Null on SCI are not relevant
On Equipment: InstallerCode missing almost everywhere, make sense to remove it. Also Many startupdate and WarrantyEndDate missing but we can keep them. For 137 equipments the WarrantyEndDate <  startupdate, doens't seem possible. They have linked approximately 900 interventions in SCH on 64 different products.
On Products: 22 product have energy type, product type e skill profile null, they are the same found on SCH. Warengroup code is useless can be discarded

